# Multi-channel CNN + LSTM cho Vietnamese Text Classification

Bài toán mặc định: **sentiment classification tiếng Việt** với 3 nhãn:

- `0`: negative
- `1`: neutral
- `2`: positive

Kiến trúc:

```text
Input sentence
→ tokenize + pad/truncate
→ embedding 200d
→ CNN channel: Conv1D kernel 3, 5, 7 + global max pooling
→ LSTM channel: LSTM hidden 128
→ concat(CNN features, LSTM feature)
→ Dense 200 + sigmoid
→ Dropout 0.2
→ Linear classifier
```

> Notebook đã có thêm phần xử lý trường hợp baseline/ablation khác đạt điểm cao hơn base model: so sánh delta với base, cảnh báo diễn giải, và tùy chọn multi-seed `mean ± std`.


## 1. Cài đặt thư viện

In [1]:
# !pip -q install torch datasets scikit-learn tqdm numpy pandas


## 2. Import và cấu hình

In [2]:
from __future__ import annotations

import json
import os
import random
import re
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"

@dataclass
class TrainConfig:
    task: str = "sentiment"          # "sentiment" hoặc "topic"
    max_features: int = 21540
    max_len: int = 400
    embedding_dim: int = 200
    num_filters: int = 150
    kernel_sizes: Tuple[int, ...] = (3, 5, 7)
    lstm_hidden: int = 128
    dense_hidden: int = 200
    dropout: float = 0.2
    batch_size: int = 64
    epochs: int = 14
    lr: float = 0.002                # Adamax default gần với Keras Adamax
    seed: int = 1337
    use_class_weights: bool = False  # đổi True nếu muốn tối ưu macro-F1 trên dữ liệu lệch lớp
    output_dir: str = "outputs_sentiment_cnn_lstm"

cfg = TrainConfig()

cfg


TrainConfig(task='sentiment', max_features=21540, max_len=400, embedding_dim=200, num_filters=150, kernel_sizes=(3, 5, 7), lstm_hidden=128, dense_hidden=200, dropout=0.2, batch_size=64, epochs=14, lr=0.002, seed=1337, output_dir='outputs_sentiment_cnn_lstm')

## 3. Reproducibility và tokenizer đơn giản cho tiếng Việt

In [3]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def basic_vi_tokenize(text: str) -> List[str]:
    text = text.lower().strip()
    text = re.sub(r"([^\w\s])", r" \1 ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split() if text else []


def build_vocab(texts: Sequence[str], max_features: int, min_freq: int = 1) -> Dict[str, int]:
    counter: Counter[str] = Counter()
    for text in texts:
        counter.update(basic_vi_tokenize(text))

    vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
    for token, freq in counter.most_common(max_features - len(vocab)):
        if freq >= min_freq:
            vocab[token] = len(vocab)
    return vocab


def encode_text(text: str, vocab: Dict[str, int], max_len: int) -> List[int]:
    ids = [vocab.get(tok, vocab[UNK_TOKEN]) for tok in basic_vi_tokenize(text)]
    ids = ids[:max_len]
    if len(ids) < max_len:
        ids += [vocab[PAD_TOKEN]] * (max_len - len(ids))
    return ids

seed_everything(cfg.seed)
print(basic_vi_tokenize("Thầy Huy và thầy Tùng dạy rất hay, nhiệt tình!"))

['thầy', 'huy', 'và', 'thầy', 'tùng', 'dạy', 'rất', 'hay', ',', 'nhiệt', 'tình', '!']


## 4. Tải dataset UIT-VSFC

In [4]:
from huggingface_hub import hf_hub_download

repo_id = "uitnlp/vietnamese_students_feedback"
revision = "refs/convert/parquet"

data_files = {
    "train": hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        revision=revision,
        filename="default/train/0000.parquet",
    ),
    "validation": hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        revision=revision,
        filename="default/validation/0000.parquet",
    ),
    "test": hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        revision=revision,
        filename="default/test/0000.parquet",
    ),
}

ds = load_dataset("parquet", data_files=data_files)
ds

default/train/0000.parquet:   0%|          | 0.00/475k [00:00<?, ?B/s]

default/validation/0000.parquet:   0%|          | 0.00/63.3k [00:00<?, ?B/s]

default/test/0000.parquet:   0%|          | 0.00/134k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'sentiment', 'topic'],
        num_rows: 11426
    })
    validation: Dataset({
        features: ['sentence', 'sentiment', 'topic'],
        num_rows: 1583
    })
    test: Dataset({
        features: ['sentence', 'sentiment', 'topic'],
        num_rows: 3166
    })
})

In [5]:
for i in range(5):
    print(ds["train"][i])

{'sentence': 'slide giáo trình đầy đủ .', 'sentiment': 2, 'topic': 1}
{'sentence': 'nhiệt tình giảng dạy , gần gũi với sinh viên .', 'sentiment': 2, 'topic': 0}
{'sentence': 'đi học đầy đủ full điểm chuyên cần .', 'sentiment': 0, 'topic': 1}
{'sentence': 'chưa áp dụng công nghệ thông tin và các thiết bị hỗ trợ cho việc giảng dạy .', 'sentiment': 0, 'topic': 0}
{'sentence': 'thầy giảng bài hay , có nhiều bài tập ví dụ ngay trên lớp .', 'sentiment': 2, 'topic': 0}


In [6]:
# sentiment: 0 negative, 1 neutral, 2 positive
# topic: 0 lecturer, 1 training_program, 2 facility, 3 others

def get_split(split: str, task: str):
    texts = list(ds[split]["sentence"])
    labels = [int(x) for x in ds[split][task]]
    return texts, labels

train_texts, train_labels = get_split("train", cfg.task)
val_texts, val_labels = get_split("validation", cfg.task)
test_texts, test_labels = get_split("test", cfg.task)

print("train:", len(train_texts))
print("validation:", len(val_texts))
print("test:", len(test_texts))
print("label distribution train:", Counter(train_labels))

train: 11426
validation: 1583
test: 3166
label distribution train: Counter({2: 5643, 0: 5325, 1: 458})


## 5. Dataset, vocab và DataLoader

In [7]:
class TextClassificationDataset(Dataset):
    def __init__(self, texts: Sequence[str], labels: Sequence[int], vocab: Dict[str, int], max_len: int):
        self.texts = list(texts)
        self.labels = list(labels)
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int):
        x = encode_text(self.texts[idx], self.vocab, self.max_len)
        y = int(self.labels[idx])
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


vocab = build_vocab(train_texts, max_features=cfg.max_features)
print("vocab_size:", len(vocab))

train_ds = TextClassificationDataset(train_texts, train_labels, vocab, cfg.max_len)
val_ds = TextClassificationDataset(val_texts, val_labels, vocab, cfg.max_len)
test_ds = TextClassificationDataset(test_texts, test_labels, vocab, cfg.max_len)


def make_loaders(batch_size: int = None, seed: int = None):
    """Tạo lại DataLoader cho mỗi lần chạy để thứ tự shuffle công bằng và reproducible."""
    if batch_size is None:
        batch_size = cfg.batch_size
    if seed is None:
        seed = cfg.seed

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        generator=generator,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = make_loaders(cfg.batch_size, cfg.seed)

x_batch, y_batch = next(iter(train_loader))
print("x_batch:", x_batch.shape)
print("y_batch:", y_batch.shape)


vocab_size: 2504
x_batch: torch.Size([64, 400])
y_batch: torch.Size([64])


## 6. Định nghĩa mô hình Multi-channel CNN + LSTM

In [8]:
class MultiChannelCnnLstm(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_classes: int,
        embedding_dim: int = 200,
        num_filters: int = 150,
        kernel_sizes: Tuple[int, ...] = (3, 7, 5),  # sát thứ tự concat Keras gốc hơn
        lstm_hidden: int = 128,
        dense_hidden: int = 200,
        dropout: float = 0.2,
        pad_idx: int = 0,
    ) -> None:
        super().__init__()

        # Keras gốc dùng 2 Embedding layer riêng
        self.embedding_cnn = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=pad_idx,
        )

        self.embedding_lstm = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=pad_idx,
        )

        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=embedding_dim,
                out_channels=num_filters,
                kernel_size=k,
            )
            for k in kernel_sizes
        ])

        self.relu = nn.ReLU()

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=lstm_hidden,
            batch_first=True,
            bidirectional=False,
        )

        concat_dim = len(kernel_sizes) * num_filters + lstm_hidden

        self.fc = nn.Linear(concat_dim, dense_hidden)
        self.fc_activation = nn.Sigmoid()
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dense_hidden, num_classes)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        # CNN channel
        emb_cnn = self.embedding_cnn(input_ids)
        cnn_in = emb_cnn.transpose(1, 2)

        pooled_outputs = []
        for conv in self.convs:
            feat = self.relu(conv(cnn_in))
            pooled = torch.max(feat, dim=2).values
            pooled_outputs.append(pooled)

        cnn_features = torch.cat(pooled_outputs, dim=1)

        # LSTM channel
        emb_lstm = self.embedding_lstm(input_ids)
        _, (h_n, _) = self.lstm(emb_lstm)
        lstm_features = h_n[-1]

        # Fusion + classifier
        features = torch.cat([lstm_features, cnn_features], dim=1)

        x = self.fc_activation(self.fc(features))
        x = self.dropout(x)

        # Return logits. Use nn.CrossEntropyLoss during training.
        logits = self.classifier(x)
        return logits

In [9]:
num_classes = 3 if cfg.task == "sentiment" else 4


def get_safe_device(min_cuda_major: int = 7) -> torch.device:
    """
    Một số runtime thấy GPU nhưng PyTorch build không support kiến trúc GPU đó
    ví dụ Tesla P100 sm_60 với PyTorch chỉ support sm_70+.
    Khi phát hiện trường hợp này, fallback sang CPU để notebook không crash.
    """
    if not torch.cuda.is_available():
        return torch.device("cpu")

    try:
        name = torch.cuda.get_device_name(0)
        major, minor = torch.cuda.get_device_capability(0)
        print(f"Detected GPU: {name}, capability: sm_{major}{minor}")

        if major < min_cuda_major:
            print("GPU này không tương thích với PyTorch CUDA hiện tại. Fallback sang CPU.")
            return torch.device("cpu")

        # Smoke test nhỏ để bắt lỗi cudaErrorNoKernelImageForDevice sớm.
        _ = torch.empty(1, device="cuda") + 1
        return torch.device("cuda")
    except Exception as exc:
        print("CUDA không dùng được trong runtime này. Fallback sang CPU.")
        print("Reason:", repr(exc))
        return torch.device("cpu")


device = get_safe_device()
seed_everything(cfg.seed)

model = MultiChannelCnnLstm(
    vocab_size=len(vocab),
    num_classes=num_classes,
    embedding_dim=cfg.embedding_dim,
    num_filters=cfg.num_filters,
    kernel_sizes=cfg.kernel_sizes,
    lstm_hidden=cfg.lstm_hidden,
    dense_hidden=cfg.dense_hidden,
    dropout=cfg.dropout,
    pad_idx=vocab[PAD_TOKEN],
).to(device)

with torch.no_grad():
    logits = model(x_batch[:4].to(device))

print(model)
print("device:", device)
print("logits shape:", logits.shape)


MultiChannelCnnLstm(
  (embedding_cnn): Embedding(2504, 200, padding_idx=0)
  (embedding_lstm): Embedding(2504, 200, padding_idx=0)
  (convs): ModuleList(
    (0): Conv1d(200, 150, kernel_size=(3,), stride=(1,))
    (1): Conv1d(200, 150, kernel_size=(5,), stride=(1,))
    (2): Conv1d(200, 150, kernel_size=(7,), stride=(1,))
  )
  (relu): ReLU()
  (lstm): LSTM(200, 128, batch_first=True)
  (fc): Linear(in_features=578, out_features=200, bias=True)
  (fc_activation): Sigmoid()
  (dropout): Dropout(p=0.2, inplace=False)
  (classifier): Linear(in_features=200, out_features=3, bias=True)
)
device: cuda
logits shape: torch.Size([4, 3])


## 7. Hàm evaluate và train

In [10]:
def make_class_weights(labels: Sequence[int], num_classes: int, device: torch.device) -> torch.Tensor:
    """
    Tính class weights theo công thức balanced:
    weight_c = n_samples / (num_classes * count_c)
    Chỉ dùng khi cfg.use_class_weights=True.
    """
    counts = Counter(int(x) for x in labels)
    total = sum(counts.values())
    weights = []
    for c in range(num_classes):
        count_c = max(1, counts.get(c, 0))
        weights.append(total / (num_classes * count_c))
    return torch.tensor(weights, dtype=torch.float32, device=device)


def make_criterion(cfg: TrainConfig, device: torch.device) -> nn.Module:
    if cfg.use_class_weights:
        weights = make_class_weights(train_labels, num_classes, device)
        print("Using class weights:", weights.detach().cpu().numpy().round(4).tolist())
        return nn.CrossEntropyLoss(weight=weights)
    return nn.CrossEntropyLoss()


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device):
    model.eval()
    criterion = make_criterion(cfg, device)
    total_loss = 0.0
    all_preds = []
    all_labels = []

    for input_ids, labels in loader:
        input_ids = input_ids.to(device)
        labels = labels.to(device)
        logits = model(input_ids)
        loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    return {
        "loss": total_loss / max(1, len(all_labels)),
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": f1_score(all_labels, all_preds, average="macro"),
        "labels": all_labels,
        "preds": all_preds,
    }


def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    cfg: TrainConfig,
    device: torch.device,
):
    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    criterion = make_criterion(cfg, device)
    optimizer = torch.optim.Adamax(model.parameters(), lr=cfg.lr)

    best_macro_f1 = -1.0
    best_path = output_dir / f"best_{cfg.task}_cnn_lstm.pt"
    history = []

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"epoch {epoch}/{cfg.epochs}")

        for input_ids, labels in pbar:
            input_ids = input_ids.to(device)
            labels = labels.to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(input_ids)
            loss = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        train_loss = running_loss / len(train_loader.dataset)
        val_metrics = evaluate(model, val_loader, device)

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
        }
        history.append(row)

        print(
            f"epoch={epoch} "
            f"train_loss={train_loss:.4f} "
            f"val_loss={val_metrics['loss']:.4f} "
            f"val_acc={val_metrics['accuracy']:.4f} "
            f"val_macro_f1={val_metrics['macro_f1']:.4f}"
        )

        if val_metrics["macro_f1"] > best_macro_f1:
            best_macro_f1 = float(val_metrics["macro_f1"])
            torch.save(model.state_dict(), best_path)
            print("saved best checkpoint ->", best_path)

    return history, best_path


## 8. Huấn luyện

In [11]:
# Reset seed và DataLoader trước khi train model chính.
# Như vậy kết quả base model không bị ảnh hưởng bởi các cell kiểm tra trước đó.
seed_everything(cfg.seed)
train_loader, val_loader, test_loader = make_loaders(cfg.batch_size, cfg.seed)

history, best_path = train_model(model, train_loader, val_loader, cfg, device)
history[-3:]


epoch 1/14:   0%|          | 0/179 [00:00<?, ?it/s]

epoch=1 train_loss=0.4202 val_loss=0.3114 val_acc=0.8907 val_macro_f1=0.6250
saved best checkpoint -> outputs_sentiment_cnn_lstm/best_sentiment_cnn_lstm.pt


epoch 2/14:   0%|          | 0/179 [00:00<?, ?it/s]

epoch=2 train_loss=0.2237 val_loss=0.2635 val_acc=0.9116 val_macro_f1=0.7417
saved best checkpoint -> outputs_sentiment_cnn_lstm/best_sentiment_cnn_lstm.pt


epoch 3/14:   0%|          | 0/179 [00:00<?, ?it/s]

...

epoch 13/14:   0%|          | 0/179 [00:00<?, ?it/s]

epoch=13 train_loss=0.0026 val_loss=0.4235 val_acc=0.9097 val_macro_f1=0.7878


epoch 14/14:   0%|          | 0/179 [00:00<?, ?it/s]

epoch=14 train_loss=0.0016 val_loss=0.4520 val_acc=0.9021 val_macro_f1=0.7621


[{'epoch': 12,
  'train_loss': 0.0029243559875057183,
  'val_loss': 0.4222188658979371,
  'val_accuracy': 0.9077700568540745,
  'val_macro_f1': 0.7596679343127118},
 {'epoch': 13,
  'train_loss': 0.002634733412443521,
  'val_loss': 0.423492938336585,
  'val_accuracy': 0.9096651926721415,
  'val_macro_f1': 0.7877978702050256},
 {'epoch': 14,
  'train_loss': 0.0016085095168189168,
  'val_loss': 0.4520059168075205,
  'val_accuracy': 0.9020846493998737,
  'val_macro_f1': 0.7620613654921188}]

## 9. Đánh giá trên test set

In [12]:
model.load_state_dict(torch.load(best_path, map_location=device))
test_metrics = evaluate(model, test_loader, device)

print("TEST")
print(f"accuracy={test_metrics['accuracy']:.4f}")
print(f"macro_f1={test_metrics['macro_f1']:.4f}")
print()
print(classification_report(test_metrics["labels"], test_metrics["preds"], digits=4))
print("confusion_matrix:")
print(confusion_matrix(test_metrics["labels"], test_metrics["preds"]))

TEST
accuracy=0.8790
macro_f1=0.7367

              precision    recall  f1-score   support

           0     0.8318    0.9652    0.8936      1409
           1     0.5446    0.3293    0.4104       167
           2     0.9566    0.8604    0.9060      1590

    accuracy                         0.8790      3166
   macro avg     0.7777    0.7183    0.7367      3166
weighted avg     0.8793    0.8790    0.8743      3166

confusion_matrix:
[[1360   20   29]
 [  79   55   33]
 [ 196   26 1368]]


## 10. So sánh với các phương pháp khác

Phần này tái hiện tinh thần thí nghiệm trong paper: không chỉ train mô hình đề xuất, mà còn so sánh với các baseline riêng lẻ và baseline truyền thống.

Các nhóm được so sánh:

- **TF-IDF + Logistic Regression**
- **TF-IDF + Linear SVM**
- **TF-IDF + Multinomial Naive Bayes**
- **CNN-only**: chỉ giữ nhánh CNN multi-filter.
- **LSTM-only**: chỉ giữ nhánh LSTM.
- **Multi-channel CNN+LSTM**: mô hình chính đã train ở trên.

Metric chính nên dùng: `macro_f1`, vì sentiment dataset thường lệch nhãn; `accuracy` vẫn được giữ để dễ đối chiếu.


In [13]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC


BASE_METHOD = "Multi-channel CNN+LSTM"


def summarize_predictions(method: str, family: str, y_true, y_pred):
    return {
        "method": method,
        "family": family,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    }


comparison_rows = []

# Thêm kết quả của mô hình chính đã train ở trên.
comparison_rows.append({
    "method": BASE_METHOD,
    "family": "Deep learning / base model",
    "accuracy": test_metrics["accuracy"],
    "macro_f1": test_metrics["macro_f1"],
    "weighted_f1": f1_score(test_metrics["labels"], test_metrics["preds"], average="weighted"),
})

comparison_rows


[{'method': 'Multi-channel CNN+LSTM',
  'family': 'Deep learning',
  'accuracy': 0.8790271636133923,
  'macro_f1': 0.7366563766351558,
  'weighted_f1': 0.8743048618385465}]

### 10.1. Baseline truyền thống: TF-IDF + ML classifier

Các baseline này thường chạy nhanh, là mốc rất tốt để kiểm tra xem deep model có thực sự đáng dùng hay không. Với dataset nhỏ/vừa, SVM hoặc Logistic Regression đôi khi rất cạnh tranh.


In [14]:
classical_models = {
    "TF-IDF + Logistic Regression": LogisticRegression(max_iter=1000, n_jobs=None),
    "TF-IDF + Linear SVM": LinearSVC(),
    "TF-IDF + MultinomialNB": MultinomialNB(),
}

for method_name, classifier in classical_models.items():
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(
            tokenizer=basic_vi_tokenize,
            token_pattern=None,
            lowercase=False,
            ngram_range=(1, 2),
            min_df=2,
            max_features=50000,
        )),
        ("clf", classifier),
    ])
    pipe.fit(train_texts, train_labels)
    preds = pipe.predict(test_texts)
    row = summarize_predictions(method_name, "TF-IDF baseline", test_labels, preds)
    comparison_rows.append(row)
    print(row)


{'method': 'TF-IDF + Logistic Regression', 'family': 'TF-IDF baseline', 'accuracy': 0.8910296904611498, 'macro_f1': 0.6684744341392718, 'weighted_f1': 0.8745003849718849}
{'method': 'TF-IDF + Linear SVM', 'family': 'TF-IDF baseline', 'accuracy': 0.8973468098547063, 'macro_f1': 0.7141728950091432, 'weighted_f1': 0.8866885599859938}
{'method': 'TF-IDF + MultinomialNB', 'family': 'TF-IDF baseline', 'accuracy': 0.861339229311434, 'macro_f1': 0.5897543830087489, 'weighted_f1': 0.8385362483650136}


### 10.2. Deep learning ablation: CNN-only và LSTM-only

Paper nhấn mạnh ý tưởng kết hợp ưu điểm của CNN và LSTM. Vì vậy ta thêm hai ablation quan trọng:

- **CNN-only** kiểm tra sức mạnh của local n-gram features.
- **LSTM-only** kiểm tra sức mạnh của sequential/context features.

Lưu ý quan trọng: một lần chạy đơn lẻ **không đủ** để kết luận mô hình nào thật sự tốt hơn. Nếu một ablation cao hơn base model một chút, đó có thể là nhiễu do seed, thứ tự shuffle, optimizer hoặc do dataset hiện tại thật sự ưu tiên đặc trưng n-gram. Phần dưới sẽ bổ sung kiểm tra chênh lệch và multi-seed để xử lý trường hợp này.


In [15]:
class TextCNNOnly(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_classes: int,
        embedding_dim: int = 200,
        num_filters: int = 150,
        kernel_sizes: Tuple[int, ...] = (3, 5, 7),
        dense_hidden: int = 200,
        dropout: float = 0.2,
        pad_idx: int = 0,
    ) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.convs = nn.ModuleList([
            nn.Conv1d(embedding_dim, num_filters, kernel_size=k)
            for k in kernel_sizes
        ])
        self.relu = nn.ReLU()
        concat_dim = len(kernel_sizes) * num_filters
        self.fc = nn.Linear(concat_dim, dense_hidden)
        self.fc_activation = nn.Sigmoid()
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dense_hidden, num_classes)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(input_ids)
        cnn_in = emb.transpose(1, 2)
        pooled_outputs = []
        for conv in self.convs:
            feat = self.relu(conv(cnn_in))
            pooled_outputs.append(torch.max(feat, dim=2).values)
        features = torch.cat(pooled_outputs, dim=1)
        x = self.fc_activation(self.fc(features))
        x = self.dropout(x)
        return self.classifier(x)


class LSTMOnly(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_classes: int,
        embedding_dim: int = 200,
        lstm_hidden: int = 128,
        dense_hidden: int = 200,
        dropout: float = 0.2,
        pad_idx: int = 0,
    ) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=lstm_hidden,
            batch_first=True,
            bidirectional=False,
        )
        self.fc = nn.Linear(lstm_hidden, dense_hidden)
        self.fc_activation = nn.Sigmoid()
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dense_hidden, num_classes)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(input_ids)
        _, (h_n, _) = self.lstm(emb)
        features = h_n[-1]
        x = self.fc_activation(self.fc(features))
        x = self.dropout(x)
        return self.classifier(x)


In [16]:
# Đặt False nếu chỉ muốn chạy baseline TF-IDF nhanh.
RUN_DEEP_COMPARISON = True
COMPARISON_EPOCHS = cfg.epochs  # đổi thành 1-3 nếu chỉ smoke test


def build_multi_channel_model() -> nn.Module:
    return MultiChannelCnnLstm(
        vocab_size=len(vocab),
        num_classes=num_classes,
        embedding_dim=cfg.embedding_dim,
        num_filters=cfg.num_filters,
        kernel_sizes=cfg.kernel_sizes,
        lstm_hidden=cfg.lstm_hidden,
        dense_hidden=cfg.dense_hidden,
        dropout=cfg.dropout,
        pad_idx=vocab[PAD_TOKEN],
    )


def build_cnn_only_model() -> nn.Module:
    return TextCNNOnly(
        vocab_size=len(vocab),
        num_classes=num_classes,
        embedding_dim=cfg.embedding_dim,
        num_filters=cfg.num_filters,
        kernel_sizes=cfg.kernel_sizes,
        dense_hidden=cfg.dense_hidden,
        dropout=cfg.dropout,
        pad_idx=vocab[PAD_TOKEN],
    )


def build_lstm_only_model() -> nn.Module:
    return LSTMOnly(
        vocab_size=len(vocab),
        num_classes=num_classes,
        embedding_dim=cfg.embedding_dim,
        lstm_hidden=cfg.lstm_hidden,
        dense_hidden=cfg.dense_hidden,
        dropout=cfg.dropout,
        pad_idx=vocab[PAD_TOKEN],
    )


def train_and_evaluate_deep_baseline(method_name: str, build_model_fn, run_seed: int = None):
    """
    Quan trọng: model phải được khởi tạo SAU khi seed được set.
    Nếu truyền sẵn model đã khởi tạo, kết quả comparison sẽ không còn thật sự reproducible.
    """
    if run_seed is None:
        run_seed = cfg.seed

    seed_everything(run_seed)
    local_train_loader, local_val_loader, local_test_loader = make_loaders(cfg.batch_size, run_seed)

    model_local = build_model_fn().to(device)

    baseline_cfg = TrainConfig(**asdict(cfg))
    baseline_cfg.epochs = COMPARISON_EPOCHS
    baseline_cfg.seed = run_seed

    safe_name = method_name.lower().replace(" ", "_").replace("+", "plus").replace("-", "_")
    baseline_cfg.output_dir = str(Path(cfg.output_dir) / "comparison" / safe_name / f"seed_{run_seed}")

    history_baseline, best_path_baseline = train_model(
        model_local,
        local_train_loader,
        local_val_loader,
        baseline_cfg,
        device,
    )

    model_local.load_state_dict(torch.load(best_path_baseline, map_location=device))
    metrics = evaluate(model_local, local_test_loader, device)

    row = {
        "method": method_name,
        "family": "Deep learning ablation",
        "seed": run_seed,
        "accuracy": metrics["accuracy"],
        "macro_f1": metrics["macro_f1"],
        "weighted_f1": f1_score(metrics["labels"], metrics["preds"], average="weighted"),
    }
    return row, history_baseline, metrics


if RUN_DEEP_COMPARISON:
    cnn_row, cnn_history, cnn_test_metrics = train_and_evaluate_deep_baseline(
        "CNN-only",
        build_cnn_only_model,
        run_seed=cfg.seed,
    )
    comparison_rows.append(cnn_row)
    print(cnn_row)

    lstm_row, lstm_history, lstm_test_metrics = train_and_evaluate_deep_baseline(
        "LSTM-only",
        build_lstm_only_model,
        run_seed=cfg.seed,
    )
    comparison_rows.append(lstm_row)
    print(lstm_row)
else:
    print("Skipped deep comparison. Set RUN_DEEP_COMPARISON = True to run CNN-only and LSTM-only.")


epoch 1/14:   0%|          | 0/179 [00:00<?, ?it/s]

epoch=1 train_loss=0.4172 val_loss=0.3031 val_acc=0.8932 val_macro_f1=0.6093
saved best checkpoint -> outputs_sentiment_cnn_lstm/comparison/cnn_only/best_sentiment_cnn_lstm.pt


epoch 2/14:   0%|          | 0/179 [00:00<?, ?it/s]

epoch=2 train_loss=0.2238 val_loss=0.2836 val_acc=0.9090 val_macro_f1=0.7452
saved best checkpoint -> outputs_sentiment_cnn_lstm/comparison/cnn_only/best_sentiment_cnn_lstm.pt


epoch 3/14:   0%|          | 0/179 [00:00<?, ?it/s]

...

epoch=12 train_loss=0.8356 val_loss=0.8474 val_acc=0.5085 val_macro_f1=0.2247


epoch 13/14:   0%|          | 0/179 [00:00<?, ?it/s]

epoch=13 train_loss=0.8352 val_loss=0.8499 val_acc=0.4454 val_macro_f1=0.2054


epoch 14/14:   0%|          | 0/179 [00:00<?, ?it/s]

epoch=14 train_loss=0.8349 val_loss=0.8520 val_acc=0.4454 val_macro_f1=0.2054
{'method': 'LSTM-only', 'family': 'Deep learning ablation', 'accuracy': 0.5022109917877448, 'macro_f1': 0.22287636669470143, 'weighted_f1': 0.33579288349138525}


### 10.3. Bảng tổng hợp kết quả

Bảng dưới đây là kết quả thực nghiệm trên cùng split train/validation/test. Khi viết báo cáo, nên trích bảng này và ghi rõ cấu hình tokenizer, max length, epoch, batch size, random seed.


In [17]:
comparison_df = pd.DataFrame(comparison_rows)

# Một số row cũ có thể chưa có cột seed, ví dụ TF-IDF hoặc base model.
if "seed" not in comparison_df.columns:
    comparison_df["seed"] = np.nan

base_macro = comparison_df.loc[comparison_df["method"] == BASE_METHOD, "macro_f1"].iloc[0]
base_acc = comparison_df.loc[comparison_df["method"] == BASE_METHOD, "accuracy"].iloc[0]

comparison_df["delta_macro_f1_vs_base"] = comparison_df["macro_f1"] - base_macro
comparison_df["delta_accuracy_vs_base"] = comparison_df["accuracy"] - base_acc
comparison_df["is_base_model"] = comparison_df["method"].eq(BASE_METHOD)

comparison_df = comparison_df.sort_values("macro_f1", ascending=False).reset_index(drop=True)
comparison_df["rank_by_macro_f1"] = np.arange(1, len(comparison_df) + 1)

# Làm tròn để bảng dễ đọc.
display(comparison_df.style.format({
    "accuracy": "{:.4f}",
    "macro_f1": "{:.4f}",
    "weighted_f1": "{:.4f}",
    "delta_macro_f1_vs_base": "{:+.4f}",
    "delta_accuracy_vs_base": "{:+.4f}",
}))

comparison_out = Path(cfg.output_dir) / "comparison_results.csv"
comparison_out.parent.mkdir(parents=True, exist_ok=True)
comparison_df.to_csv(comparison_out, index=False)
print("saved comparison table ->", comparison_out.resolve())


def explain_if_base_is_not_best(df: pd.DataFrame, base_method: str = BASE_METHOD, metric: str = "macro_f1"):
    best = df.iloc[0]
    base = df[df["method"] == base_method].iloc[0]
    diff = float(best[metric] - base[metric])

    print("\nKẾT LUẬN TỰ ĐỘNG")
    print("-" * 80)

    if best["method"] == base_method:
        print(f"Base model đang đứng đầu theo {metric}. Có thể báo cáo đây là model tốt nhất trong lần chạy này.")
        return

    print(f"Base model KHÔNG đứng đầu theo {metric} trong lần chạy này.")
    print(f"Best method: {best['method']} | {metric}={best[metric]:.4f}")
    print(f"Base method: {base_method} | {metric}={base[metric]:.4f}")
    print(f"Chênh lệch: {diff:+.4f}")

    if abs(diff) < 0.005:
        print("Chênh lệch rất nhỏ (<0.5 điểm %). Không nên kết luận method kia thật sự tốt hơn nếu chưa chạy multi-seed.")
    elif abs(diff) < 0.01:
        print("Chênh lệch nhỏ (<1 điểm %). Nên báo cáo là hai mô hình xấp xỉ nhau và cần multi-seed để kết luận.")
    else:
        print("Chênh lệch đáng kể hơn. Nếu kết quả này lặp lại qua nhiều seed, hãy báo cáo trung thực rằng method kia tốt hơn trên thiết lập/dataset này.")

    print("\nCách viết báo cáo an toàn:")
    print(
        f"Trong thí nghiệm một seed, {best['method']} đạt {metric} cao nhất. "
        f"Multi-channel CNN+LSTM không luôn vượt mọi baseline trên UIT-VSFC; "
        f"cần đánh giá thêm multi-seed để kết luận độ ổn định."
    )


explain_if_base_is_not_best(comparison_df)


,method,family,accuracy,macro_f1,weighted_f1
0,Multi-channel CNN+LSTM,Deep learning,0.8790,0.7367,0.8743
1,CNN-only,Deep learning ablation,0.8857,0.7142,0.8776
2,TF-IDF + Linear SVM,TF-IDF baseline,0.8973,0.7142,0.8867
3,TF-IDF + Logistic Regression,TF-IDF baseline,0.8910,0.6685,0.8745
4,TF-IDF + MultinomialNB,TF-IDF baseline,0.8613,0.5898,0.8385
5,LSTM-only,Deep learning ablation,0.5022,0.2229,0.3358


saved comparison table -> /kaggle/working/outputs_sentiment_cnn_lstm/comparison_results.csv


### 10.4. Multi-seed comparison để xử lý khi model khác tốt hơn base

Nếu một phiên bản khác tốt hơn base model trong một lần chạy, không nên sửa số liệu thủ công. Cách xử lý đúng là chạy nhiều seed rồi báo cáo **mean ± std**.

Cell dưới đây mặc định để `RUN_MULTI_SEED_COMPARISON = False` để tránh tốn thời gian. Khi cần kết quả chắc chắn cho báo cáo, đổi thành `True`.


In [ ]:
RUN_MULTI_SEED_COMPARISON = False
MULTI_SEED_LIST = [1337, 2024, 3407]
MULTI_SEED_EPOCHS = cfg.epochs

deep_model_builders = {
    BASE_METHOD: build_multi_channel_model,
    "CNN-only": build_cnn_only_model,
    "LSTM-only": build_lstm_only_model,
}


def run_multi_seed_deep_comparison():
    rows = []

    old_epochs = COMPARISON_EPOCHS
    try:
        globals()["COMPARISON_EPOCHS"] = MULTI_SEED_EPOCHS

        for seed in MULTI_SEED_LIST:
            for method_name, builder in deep_model_builders.items():
                print("\n" + "=" * 100)
                print(f"Running {method_name} | seed={seed}")
                print("=" * 100)

                row, _, _ = train_and_evaluate_deep_baseline(
                    method_name=method_name,
                    build_model_fn=builder,
                    run_seed=seed,
                )
                row["family"] = "Deep learning multi-seed"
                rows.append(row)

    finally:
        globals()["COMPARISON_EPOCHS"] = old_epochs

    runs_df = pd.DataFrame(rows)

    summary_df = (
        runs_df
        .groupby("method", as_index=False)
        .agg(
            accuracy_mean=("accuracy", "mean"),
            accuracy_std=("accuracy", "std"),
            macro_f1_mean=("macro_f1", "mean"),
            macro_f1_std=("macro_f1", "std"),
            weighted_f1_mean=("weighted_f1", "mean"),
            weighted_f1_std=("weighted_f1", "std"),
        )
        .sort_values("macro_f1_mean", ascending=False)
        .reset_index(drop=True)
    )

    base_mean = summary_df.loc[summary_df["method"] == BASE_METHOD, "macro_f1_mean"].iloc[0]
    summary_df["delta_macro_f1_mean_vs_base"] = summary_df["macro_f1_mean"] - base_mean

    out_dir = Path(cfg.output_dir)
    runs_path = out_dir / "deep_multi_seed_runs.csv"
    summary_path = out_dir / "deep_multi_seed_summary.csv"
    runs_df.to_csv(runs_path, index=False)
    summary_df.to_csv(summary_path, index=False)

    print("saved runs ->", runs_path.resolve())
    print("saved summary ->", summary_path.resolve())

    display(summary_df.style.format({
        "accuracy_mean": "{:.4f}",
        "accuracy_std": "{:.4f}",
        "macro_f1_mean": "{:.4f}",
        "macro_f1_std": "{:.4f}",
        "weighted_f1_mean": "{:.4f}",
        "weighted_f1_std": "{:.4f}",
        "delta_macro_f1_mean_vs_base": "{:+.4f}",
    }))

    return runs_df, summary_df


if RUN_MULTI_SEED_COMPARISON:
    multi_seed_runs_df, multi_seed_summary_df = run_multi_seed_deep_comparison()
else:
    print("Skipped multi-seed comparison. Set RUN_MULTI_SEED_COMPARISON = True for robust reporting.")


## 11. Lưu config, vocab và report


In [18]:
output_dir = Path(cfg.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(asdict(cfg) | {"vocab_size": len(vocab), "num_classes": num_classes}, f, indent=2, ensure_ascii=False)

with open(output_dir / "vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False)

report = classification_report(test_metrics["labels"], test_metrics["preds"], digits=4)
with open(output_dir / "test_report.txt", "w", encoding="utf-8") as f:
    f.write(f"accuracy={test_metrics['accuracy']:.4f}\n")
    f.write(f"macro_f1={test_metrics['macro_f1']:.4f}\n\n")
    f.write(report)

print("saved to", output_dir.resolve())

saved to /kaggle/working/outputs_sentiment_cnn_lstm


## 12. Inference thử với câu mới


In [19]:
label_names = {
    "sentiment": ["negative", "neutral", "positive"],
    "topic": ["lecturer", "training_program", "facility", "others"],
}

@torch.no_grad()
def predict(texts: List[str]):
    model.eval()
    input_ids = torch.tensor([encode_text(t, vocab, cfg.max_len) for t in texts], dtype=torch.long).to(device)
    logits = model(input_ids)
    probs = torch.softmax(logits, dim=1).cpu().numpy()
    preds = probs.argmax(axis=1)
    results = []
    for text, pred, prob in zip(texts, preds, probs):
        results.append({
            "text": text,
            "pred_id": int(pred),
            "pred_label": label_names[cfg.task][int(pred)],
            "confidence": float(prob[int(pred)]),
        })
    return results

samples = [
    "giảng viên dạy rất dễ hiểu và nhiệt tình .",
    "phòng học nóng và thiết bị quá cũ .",
    "em chưa có ý kiến về môn học này .",
]

predict(samples)

[{'text': 'giảng viên dạy rất dễ hiểu và nhiệt tình .',
  'pred_id': 2,
  'pred_label': 'positive',
  'confidence': 0.9999531507492065},
 {'text': 'phòng học nóng và thiết bị quá cũ .',
  'pred_id': 0,
  'pred_label': 'negative',
  'confidence': 0.9998680353164673},
 {'text': 'em chưa có ý kiến về môn học này .',
  'pred_id': 1,
  'pred_label': 'neutral',
  'confidence': 0.6054686307907104}]